# Daily Challenge — MCP + Airbnb Mini-Agent

## Complete and thoroughly commented solution

This notebook combines two MCP servers:

- a local notes server created with Python `FastMCP`;
- an Airbnb server, using either an offline stub or the real OpenBnB npm
  package.

A deterministic planner is enabled by default, so the complete notebook can
run without tokens, npm, or external network calls. An optional GitHub
Models planner is included for experimentation.

## End-to-end architecture

```text
User prompt
   │
   ├── discover tools from LocalNotes MCP
   ├── discover tools from Airbnb MCP
   │
   ↓
Prefix + convert schemas
   notes__add_note
   airbnb__airbnb_search
   │
   ↓
Stub or real LLM planner
   │
   ↓
Prefixed tool_calls
   │
   ├── route notes__*   → notes MCP session
   └── route airbnb__* → Airbnb MCP session
   │
   ↓
Raw tool results + readable final answer
```

## Learning objectives

You will learn how to:

1. run two MCP servers over STDIO;
2. discover tools from both sessions;
3. prefix names to prevent collisions;
4. convert MCP schemas to LLM function specifications;
5. plan multiple tool calls from one prompt;
6. route calls back to the correct server;
7. keep stub mode deterministic and token-free;
8. opt into a real Airbnb server and GitHub Models;
9. inspect calls and results before generating a final answer.

# 1. Install dependencies

In [ ]:
# Pin MCP to the current stable 1.x line. Version 2 is still pre-release at
# the date of this exercise, so `<2` prevents a breaking upgrade.
#
# `nest_asyncio` helps in notebook environments with an existing event loop.
# `requests` is used only by the optional GitHub Models planner.

%pip install -qU \
    "mcp[cli]>=1.27,<2" \
    "nest_asyncio>=1.6,<2" \
    "requests>=2.31,<3"

In [ ]:
# Optional npm installation for the real OpenBnB server.
#
# Leave this False for the default offline demonstration. The npx command
# can download the package on demand when real mode is enabled.

INSTALL_REAL_AIRBNB_PACKAGE = False

if INSTALL_REAL_AIRBNB_PACKAGE:
    !npm install -g @openbnb/mcp-server-airbnb
else:
    print("Skipping npm installation; offline Airbnb stub will be used.")

In [ ]:
# Core imports and version checks.

import asyncio
import copy
import importlib.metadata as metadata
import json
import os
import py_compile
import re
import shutil
import subprocess
import sys
from pathlib import Path
from typing import Any

import nest_asyncio
import requests
from mcp import ClientSession, StdioServerParameters, types
from mcp.client.stdio import stdio_client

# Allow nested asynchronous execution inside Colab/Jupyter.
nest_asyncio.apply()

print("Python:", sys.version.split()[0])
print("MCP SDK:", metadata.version("mcp"))
print("npx available:", bool(shutil.which("npx")))

assert sys.version_info >= (3, 10)

# 2. Configuration

In [ ]:
# Safe defaults: everything works locally with no credentials.
USE_REAL_AIRBNB = False
USE_REAL_LLM = False

# Respect robots.txt by default. Change this only for explicit testing.
IGNORE_ROBOTS_TXT = False

# Optional token forwarded to child MCP processes when supplied.
MCP_HTTP_TOKEN = os.getenv("MCP_HTTP_TOKEN", "")

# Optional GitHub Models settings.
GITHUB_MODEL = os.getenv(
    "GITHUB_MODEL",
    "openai/gpt-4.1",
)

LOCAL_SERVER = Path("local_notes_server.py").resolve()
AIRBNB_STUB_SERVER = Path("airbnb_stub_server.py").resolve()

print("Real Airbnb server:", USE_REAL_AIRBNB)
print("Real LLM planner:", USE_REAL_LLM)
print("MCP HTTP token configured:", bool(MCP_HTTP_TOKEN))

## Optional Colab secret

The following cell reads `GITHUB_TOKEN` only when real LLM mode is enabled.
Stub mode does not need any secret.

In [ ]:
if USE_REAL_LLM and not os.getenv("GITHUB_TOKEN"):
    try:
        from google.colab import userdata

        github_token = userdata.get("GITHUB_TOKEN")
        if github_token:
            os.environ["GITHUB_TOKEN"] = github_token
    except Exception:
        # Expected outside Google Colab or when no secret exists.
        pass

print(
    "GITHUB_TOKEN available:",
    bool(os.getenv("GITHUB_TOKEN")),
)

# 3. Create the local notes MCP server

In [ ]:
%%writefile local_notes_server.py
"""Local MCP notes server used by the mini-agent.

The server keeps notes in memory for the lifetime of one MCP process.
It exposes two tools:
- add_note(text)
- list_notes()

No database or LLM is involved.
"""

from mcp.server.fastmcp import FastMCP


# FastMCP converts the Python signatures and docstrings into MCP schemas.
mcp = FastMCP("LocalNotes")

# The list is intentionally in memory for a small educational example.
notes: list[str] = []


@mcp.tool()
def add_note(text: str) -> str:
    """Save one text note in memory.

    Args:
        text: The note that should be remembered.

    Returns:
        A confirmation containing the note number and saved text.
    """
    cleaned_text = text.strip()

    if not cleaned_text:
        return "The note was empty and was not saved."

    notes.append(cleaned_text)
    return f"Saved note #{len(notes)}: {cleaned_text}"


@mcp.tool()
def list_notes() -> str:
    """Return all notes saved during the current server session."""
    if not notes:
        return "No notes yet."

    # Use real line breaks so the result is easy to read.
    return "\n".join(
        f"{index}. {note}"
        for index, note in enumerate(notes, start=1)
    )


def main() -> None:
    """Start the notes server over STDIO."""
    mcp.run(transport="stdio")


if __name__ == "__main__":
    main()

The notes server is stateful only while its process is alive. It provides:

- `add_note(text)` to append a note;
- `list_notes()` to read all notes saved in that session.

In production, persistence would use a database rather than a Python list.

# 4. Create the offline Airbnb MCP stub

In [ ]:
%%writefile airbnb_stub_server.py
"""Offline Airbnb-like MCP stub for deterministic notebook execution.

The real OpenBnB server performs network searches. This local server exposes
compatible tool names and returns fixed listings, allowing the mini-agent to
run without Node.js, npm, internet access, or external credentials.
"""

from typing import Any

from mcp.server.fastmcp import FastMCP


mcp = FastMCP("AirbnbStub")


# Fixed sample data. Prices are demonstration values, not live quotations.
LISTINGS: dict[str, list[dict[str, Any]]] = {
    "paris": [
        {
            "id": "paris-101",
            "name": "Canal-side studio",
            "location": "Paris, France",
            "price_per_night": 145,
            "rating": 4.82,
            "features": ["balcony", "wifi", "kitchen"],
            "url": "https://example.com/airbnb/paris-101",
        },
        {
            "id": "paris-202",
            "name": "Montmartre apartment",
            "location": "Paris, France",
            "price_per_night": 210,
            "rating": 4.91,
            "features": ["city view", "workspace", "washer"],
            "url": "https://example.com/airbnb/paris-202",
        },
        {
            "id": "paris-303",
            "name": "Latin Quarter room",
            "location": "Paris, France",
            "price_per_night": 95,
            "rating": 4.67,
            "features": ["private room", "wifi"],
            "url": "https://example.com/airbnb/paris-303",
        },
    ],
    "london": [
        {
            "id": "london-101",
            "name": "Shoreditch loft",
            "location": "London, UK",
            "price_per_night": 190,
            "rating": 4.79,
            "features": ["workspace", "wifi", "kitchen"],
            "url": "https://example.com/airbnb/london-101",
        },
        {
            "id": "london-202",
            "name": "Camden garden flat",
            "location": "London, UK",
            "price_per_night": 230,
            "rating": 4.88,
            "features": ["garden", "washer", "kitchen"],
            "url": "https://example.com/airbnb/london-202",
        },
    ],
    "abidjan": [
        {
            "id": "abidjan-101",
            "name": "Cocody modern residence",
            "location": "Abidjan, Côte d’Ivoire",
            "price_per_night": 120,
            "rating": 4.74,
            "features": ["pool", "wifi", "parking"],
            "url": "https://example.com/airbnb/abidjan-101",
        },
        {
            "id": "abidjan-202",
            "name": "Marcory business apartment",
            "location": "Abidjan, Côte d’Ivoire",
            "price_per_night": 85,
            "rating": 4.69,
            "features": ["workspace", "wifi", "air conditioning"],
            "url": "https://example.com/airbnb/abidjan-202",
        },
    ],
}


def normalize_location(location: str) -> str:
    """Map common location strings to the stub's lookup keys."""
    normalized = location.strip().lower()

    if "paris" in normalized:
        return "paris"
    if "london" in normalized:
        return "london"
    if "abidjan" in normalized:
        return "abidjan"

    return normalized


@mcp.tool()
def airbnb_search(
    location: str,
    checkin: str | None = None,
    checkout: str | None = None,
    adults: int = 1,
    children: int = 0,
    infants: int = 0,
    pets: int = 0,
    minPrice: int | None = None,
    maxPrice: int | None = None,
    propertyType: str | None = None,
    limit: int = 3,
) -> dict[str, Any]:
    """Return fixed Airbnb-like listings for a location.

    The argument names intentionally resemble the real OpenBnB MCP tool so
    that the orchestrator can switch between the stub and real server.
    """
    key = normalize_location(location)
    candidates = list(LISTINGS.get(key, []))

    if minPrice is not None:
        candidates = [
            item for item in candidates
            if item["price_per_night"] >= minPrice
        ]

    if maxPrice is not None:
        candidates = [
            item for item in candidates
            if item["price_per_night"] <= maxPrice
        ]

    # The stub does not deeply model guest capacity or property types, but
    # it echoes filters to make the data flow visible.
    selected = candidates[: max(1, min(limit, 5))]

    return {
        "source": "offline Airbnb stub",
        "location": location,
        "checkin": checkin,
        "checkout": checkout,
        "guests": {
            "adults": adults,
            "children": children,
            "infants": infants,
            "pets": pets,
        },
        "propertyType": propertyType,
        "count": len(selected),
        "listings": selected,
    }


@mcp.tool()
def airbnb_listing_details(id: str) -> dict[str, Any]:
    """Return one fixed listing by its identifier."""
    for city_listings in LISTINGS.values():
        for listing in city_listings:
            if listing["id"] == id:
                return {
                    "source": "offline Airbnb stub",
                    "listing": listing,
                }

    return {
        "error": f"Unknown listing id: {id}",
    }


def main() -> None:
    """Start the offline Airbnb stub over STDIO."""
    mcp.run(transport="stdio")


if __name__ == "__main__":
    main()

## Why the stub mirrors real tool names

The stub exposes:

- `airbnb_search`;
- `airbnb_listing_details`.

These match the real OpenBnB server's main tool names. The orchestrator can
therefore switch servers without changing its routing strategy.

Stub prices and URLs are fictional demonstration data, not live Airbnb
results.

In [ ]:
# Validate both server files before opening STDIO sessions.

for filename in [
    "local_notes_server.py",
    "airbnb_stub_server.py",
]:
    py_compile.compile(filename, doraise=True)
    print(f"{filename} syntax: OK")

# 5. Configure STDIO server processes

In [ ]:
def find_command(name: str) -> str:
    """Locate an executable or raise a clear setup error."""
    command = shutil.which(name)

    if command is None:
        raise RuntimeError(
            f"Required command {name!r} was not found in PATH."
        )

    return command


def build_base_env() -> dict[str, str]:
    """Forward the environment and add the optional MCP token."""
    environment = os.environ.copy()

    if MCP_HTTP_TOKEN:
        environment["MCP_HTTP_TOKEN"] = MCP_HTTP_TOKEN

    return environment


def notes_server_params() -> StdioServerParameters:
    """Return the local notes server child-process configuration."""
    return StdioServerParameters(
        command=find_command("mcp"),
        args=["run", str(LOCAL_SERVER)],
        env=build_base_env(),
    )


def airbnb_server_params() -> StdioServerParameters:
    """Return stub or real Airbnb server parameters."""
    if not USE_REAL_AIRBNB:
        return StdioServerParameters(
            command=find_command("mcp"),
            args=["run", str(AIRBNB_STUB_SERVER)],
            env=build_base_env(),
        )

    # OpenBnB documents npx -y as the normal local startup command.
    arguments = [
        "-y",
        "@openbnb/mcp-server-airbnb",
    ]

    if IGNORE_ROBOTS_TXT:
        arguments.append("--ignore-robots-txt")

    return StdioServerParameters(
        command=find_command("npx"),
        args=arguments,
        env=build_base_env(),
    )

# 6. Convert MCP tools into prefixed LLM functions

In [ ]:
def get_input_schema(tool: Any) -> dict[str, Any]:
    """Read inputSchema across compatible MCP SDK naming styles."""
    schema = getattr(tool, "inputSchema", None)

    if schema is None:
        schema = getattr(tool, "input_schema", None)

    return copy.deepcopy(schema or {
        "type": "object",
        "properties": {},
    })


def convert_tool(
    tool: Any,
    prefix: str,
) -> dict[str, Any]:
    """Wrap one MCP schema in the function format expected by an LLM."""
    return {
        "type": "function",
        "function": {
            # Prefixes identify the owning server and avoid collisions.
            "name": f"{prefix}__{tool.name}",
            "description": (
                f"[{prefix} MCP server] "
                f"{tool.description or 'MCP tool'}"
            ),
            # Preserve the complete JSON Schema, including required fields.
            "parameters": get_input_schema(tool),
        },
    }

# 7. Deterministic stub planner

In [ ]:
def available_function_names(
    functions: list[dict[str, Any]],
) -> set[str]:
    """Return all prefixed functions visible to the planner."""
    return {
        item["function"]["name"]
        for item in functions
    }


def extract_city(prompt: str) -> str:
    """Extract a common demonstration city from the prompt."""
    lower_prompt = prompt.lower()

    for city in ["Paris", "London", "Abidjan"]:
        if city.lower() in lower_prompt:
            return city

    return "Paris"


def extract_integer(
    pattern: str,
    prompt: str,
    default: int | None = None,
) -> int | None:
    """Extract the first integer captured by a regular expression."""
    match = re.search(pattern, prompt, flags=re.IGNORECASE)
    return int(match.group(1)) if match else default


def extract_note(prompt: str) -> str | None:
    """Extract text after phrases such as 'save a note that'."""
    match = re.search(
        r"(?:save|add|write|remember)\s+(?:a\s+)?note"
        r"(?:\s+that|\s*:)?\s+(.+)$",
        prompt,
        flags=re.IGNORECASE,
    )

    return match.group(1).strip(" .") if match else None

In [ ]:
def stub_plan(
    prompt: str,
    functions: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    """Plan an Airbnb search plus optional note operations."""
    available = available_function_names(functions)
    lower_prompt = prompt.lower()
    calls: list[dict[str, Any]] = []

    if any(
        word in lower_prompt
        for word in [
            "airbnb",
            "stay",
            "stays",
            "listing",
            "listings",
            "apartment",
            "accommodation",
        ]
    ):
        function_name = "airbnb__airbnb_search"

        if function_name not in available:
            raise ValueError(
                "airbnb_search was not discovered from the Airbnb server."
            )

        arguments: dict[str, Any] = {
            "location": extract_city(prompt),
            "adults": extract_integer(
                r"(\d+)\s+adults?",
                prompt,
                default=1,
            ),
        }

        max_price = extract_integer(
            r"(?:under|below|max(?:imum)?(?:\s+price)?(?:\s+of)?)"
            r"\s*[$€£]?\s*(\d+)",
            prompt,
        )

        if max_price is not None:
            arguments["maxPrice"] = max_price

        # The local stub accepts a small result limit. Avoid sending this
        # non-standard argument to the real OpenBnB server.
        if not USE_REAL_AIRBNB:
            arguments["limit"] = extract_integer(
                r"(?:find|show|give|return)\s+(\d+)\s+",
                prompt,
                default=2,
            )

        calls.append({
            "name": function_name,
            "args": arguments,
        })

    note_text = extract_note(prompt)

    if note_text:
        function_name = "notes__add_note"

        if function_name not in available:
            raise ValueError(
                "add_note was not discovered from the notes server."
            )

        calls.append({
            "name": function_name,
            "args": {"text": note_text},
        })

    if "list note" in lower_prompt or "show note" in lower_prompt:
        function_name = "notes__list_notes"

        if function_name in available:
            calls.append({
                "name": function_name,
                "args": {},
            })

    if not calls:
        raise ValueError(
            "The stub planner could not identify a supported task."
        )

    return calls

# 8. Optional GitHub Models planner

In [ ]:
GITHUB_MODELS_URL = (
    "https://models.github.ai/inference/chat/completions"
)
GITHUB_API_VERSION = "2026-03-10"


def github_models_plan(
    prompt: str,
    functions: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    """Ask GitHub Models to propose calls from both MCP servers."""
    token = os.getenv("GITHUB_TOKEN", "").strip()

    if not token:
        raise RuntimeError(
            "Set GITHUB_TOKEN or keep USE_REAL_LLM=False."
        )

    response = requests.post(
        GITHUB_MODELS_URL,
        headers={
            "Accept": "application/vnd.github+json",
            "Authorization": f"Bearer {token}",
            "X-GitHub-Api-Version": GITHUB_API_VERSION,
            "Content-Type": "application/json",
        },
        json={
            "model": GITHUB_MODEL,
            "messages": [
                {
                    "role": "system",
                    "content": (
                        "Plan every tool call needed for the request. "
                        "You may call functions from both MCP servers. "
                        "Return tool calls only."
                    ),
                },
                {"role": "user", "content": prompt},
            ],
            "tools": functions,
            "tool_choice": "auto",
            "temperature": 0,
            "max_tokens": 500,
        },
        timeout=60,
    )
    response.raise_for_status()

    message = response.json()["choices"][0]["message"]
    raw_calls = message.get("tool_calls") or []
    calls: list[dict[str, Any]] = []

    for raw_call in raw_calls:
        function = raw_call["function"]
        raw_arguments = function.get("arguments", {})
        arguments = (
            json.loads(raw_arguments)
            if isinstance(raw_arguments, str)
            else raw_arguments
        )

        calls.append({
            "name": function["name"],
            "args": arguments,
        })

    if not calls:
        raise RuntimeError("The real LLM returned no tool calls.")

    return calls


def plan_tool_calls(
    prompt: str,
    functions: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    """Select the configured planner."""
    if USE_REAL_LLM:
        return github_models_plan(prompt, functions)

    return stub_plan(prompt, functions)

# 9. Result parsing and safe routing

In [ ]:
def extract_tool_value(result: Any) -> Any:
    """Extract structured content first, then fall back to text blocks."""
    structured = getattr(result, "structuredContent", None)

    if structured is None:
        structured = getattr(result, "structured_content", None)

    if isinstance(structured, dict):
        if set(structured) == {"result"}:
            return structured["result"]
        return structured

    texts: list[str] = []

    for block in getattr(result, "content", []):
        if isinstance(block, types.TextContent):
            texts.append(block.text)
        elif getattr(block, "text", None) is not None:
            texts.append(str(block.text))

    if len(texts) == 1:
        try:
            return json.loads(texts[0])
        except json.JSONDecodeError:
            return texts[0]

    return texts


def validate_call(
    call: dict[str, Any],
    prefixed_tools: dict[str, tuple[str, str]],
) -> None:
    """Validate the planner output before any MCP execution."""
    if not isinstance(call, dict):
        raise TypeError("Every tool call must be a dictionary.")

    if call.get("name") not in prefixed_tools:
        raise ValueError(
            f"Unknown prefixed tool: {call.get('name')!r}"
        )

    if not isinstance(call.get("args"), dict):
        raise TypeError("Tool arguments must be a dictionary.")

# 10. Orchestrate both MCP servers

In [ ]:
def compact_listing_summary(value: Any) -> str:
    """Render stub listings into a small readable answer."""
    if not isinstance(value, dict):
        return str(value)

    listings = value.get("listings")

    if not isinstance(listings, list):
        return json.dumps(value, ensure_ascii=False)

    if not listings:
        return f"No listings found for {value.get('location')}."

    lines = [f"Listings for {value.get('location')}:"]

    for item in listings:
        lines.append(
            "- "
            f"{item.get('name')} — "
            f"{item.get('price_per_night')} per night — "
            f"rating {item.get('rating')} — "
            f"{item.get('url')}"
        )

    return "\n".join(lines)

In [ ]:
async def orchestrate(prompt: str) -> dict[str, Any]:
    """Discover, plan, route, execute, and print the complete flow."""
    async with stdio_client(notes_server_params()) as (
        notes_read,
        notes_write,
    ):
        async with ClientSession(
            notes_read,
            notes_write,
        ) as notes_session:
            await notes_session.initialize()
            notes_tools_result = await notes_session.list_tools()

            async with stdio_client(airbnb_server_params()) as (
                airbnb_read,
                airbnb_write,
            ):
                async with ClientSession(
                    airbnb_read,
                    airbnb_write,
                ) as airbnb_session:
                    await airbnb_session.initialize()
                    airbnb_tools_result = await airbnb_session.list_tools()

                    functions = [
                        *[
                            convert_tool(tool, "notes")
                            for tool in notes_tools_result.tools
                        ],
                        *[
                            convert_tool(tool, "airbnb")
                            for tool in airbnb_tools_result.tools
                        ],
                    ]

                    # Map prefixed LLM names back to server and MCP name.
                    prefixed_tools = {}

                    for tool in notes_tools_result.tools:
                        prefixed_tools[
                            f"notes__{tool.name}"
                        ] = ("notes", tool.name)

                    for tool in airbnb_tools_result.tools:
                        prefixed_tools[
                            f"airbnb__{tool.name}"
                        ] = ("airbnb", tool.name)

                    calls = plan_tool_calls(prompt, functions)
                    results = []

                    print("Notes tools:", [
                        tool.name for tool in notes_tools_result.tools
                    ])
                    print("Airbnb tools:", [
                        tool.name for tool in airbnb_tools_result.tools
                    ])
                    print("LLM/stub function names:", [
                        item["function"]["name"]
                        for item in functions
                    ])
                    print(
                        "Planner:",
                        "GitHub Models"
                        if USE_REAL_LLM
                        else "deterministic stub",
                    )
                    print("Prompt:", prompt)
                    print("tool_calls:", calls)

                    for call in calls:
                        validate_call(call, prefixed_tools)
                        prefix, original_name = prefixed_tools[
                            call["name"]
                        ]
                        session = (
                            notes_session
                            if prefix == "notes"
                            else airbnb_session
                        )

                        raw_result = await session.call_tool(
                            original_name,
                            arguments=call["args"],
                        )
                        value = extract_tool_value(raw_result)
                        record = {
                            "name": call["name"],
                            "args": call["args"],
                            "value": value,
                        }
                        results.append(record)
                        print("tool_result:", record)

                    answer_parts = []

                    for record in results:
                        if record["name"].startswith("airbnb__"):
                            answer_parts.append(
                                compact_listing_summary(record["value"])
                            )
                        elif record["name"] == "notes__add_note":
                            answer_parts.append(
                                f"Note: {record['value']}"
                            )
                        elif record["name"] == "notes__list_notes":
                            answer_parts.append(
                                f"Saved notes:\n{record['value']}"
                            )

                    final_answer = "\n\n".join(answer_parts)

                    print("\nFINAL ANSWER")
                    print(final_answer)

                    return {
                        "prompt": prompt,
                        "functions": functions,
                        "tool_calls": calls,
                        "tool_results": results,
                        "answer": final_answer,
                    }

# 11. Run the complete stub demonstration

In [ ]:
# Tweak the city, number of adults, maximum price, or note text here.
prompt = (
    "Find 2 Airbnb stays in Paris for 2 adults under $250 per night "
    "and save a note that I prefer a balcony."
)

demo_result = await orchestrate(prompt)

In [ ]:
# Automated checks confirm that both servers were used.

called_names = {
    call["name"]
    for call in demo_result["tool_calls"]
}

assert "airbnb__airbnb_search" in called_names
assert "notes__add_note" in called_names
assert "Listings for Paris" in demo_result["answer"]
assert "Saved note #1" in demo_result["answer"]

print("End-to-end assertions: OK")

# 12. Try another city and list the saved notes

In [ ]:
# Each orchestrate() invocation starts fresh server processes, so note memory
# is scoped to that invocation. This prompt both saves and lists one note.

second_prompt = (
    "Show 2 stays in Abidjan for 1 adult under $150, "
    "save a note that I need reliable wifi, and list notes."
)

second_result = await orchestrate(second_prompt)

# 13. Create a standalone agent script

In [ ]:
%%writefile mcp_airbnb_agent.py
"""Two-server MCP mini-agent with stub or optional real planning.

The agent connects to:
- local_notes_server.py
- airbnb_stub_server.py, or the real OpenBnB npm MCP server

It discovers tool schemas dynamically, prefixes tool names to avoid name
collisions, asks a deterministic stub or GitHub Models to plan calls, then
routes every call back to the correct MCP session.
"""

import asyncio
import copy
import json
import os
import re
import shutil
from pathlib import Path
from typing import Any

import requests
from mcp import ClientSession, StdioServerParameters, types
from mcp.client.stdio import stdio_client


BASE_DIR = Path(__file__).resolve().parent
NOTES_SERVER = BASE_DIR / "local_notes_server.py"
AIRBNB_STUB_SERVER = BASE_DIR / "airbnb_stub_server.py"

USE_REAL_AIRBNB = os.getenv(
    "USE_REAL_AIRBNB",
    "false",
).lower() == "true"

USE_REAL_LLM = os.getenv(
    "USE_REAL_LLM",
    "false",
).lower() == "true"

IGNORE_ROBOTS_TXT = os.getenv(
    "IGNORE_ROBOTS_TXT",
    "false",
).lower() == "true"

GITHUB_MODELS_URL = (
    "https://models.github.ai/inference/chat/completions"
)
GITHUB_API_VERSION = "2026-03-10"
GITHUB_MODEL = os.getenv(
    "GITHUB_MODEL",
    "openai/gpt-4.1",
)


def find_command(name: str) -> str:
    """Return an executable path or raise a clear setup error."""
    command = shutil.which(name)

    if command is None:
        raise RuntimeError(
            f"Required command {name!r} was not found in PATH."
        )

    return command


def build_base_env() -> dict[str, str]:
    """Forward the environment and optionally add an MCP HTTP token."""
    environment = os.environ.copy()
    token = os.getenv("MCP_HTTP_TOKEN", "").strip()

    if token:
        environment["MCP_HTTP_TOKEN"] = token

    return environment


def notes_server_params() -> StdioServerParameters:
    """Describe how to start the local notes MCP server."""
    return StdioServerParameters(
        command=find_command("mcp"),
        args=["run", str(NOTES_SERVER)],
        env=build_base_env(),
    )


def airbnb_server_params() -> StdioServerParameters:
    """Select the offline stub or the real OpenBnB npm server."""
    if not USE_REAL_AIRBNB:
        return StdioServerParameters(
            command=find_command("mcp"),
            args=["run", str(AIRBNB_STUB_SERVER)],
            env=build_base_env(),
        )

    # The official package documentation recommends npx -y.
    arguments = [
        "-y",
        "@openbnb/mcp-server-airbnb",
    ]

    # Respect robots.txt by default. This opt-in flag is for explicit tests.
    if IGNORE_ROBOTS_TXT:
        arguments.append("--ignore-robots-txt")

    return StdioServerParameters(
        command=find_command("npx"),
        args=arguments,
        env=build_base_env(),
    )


def get_input_schema(tool: Any) -> dict[str, Any]:
    """Read a tool schema across MCP SDK field naming variations."""
    schema = getattr(tool, "inputSchema", None)

    if schema is None:
        schema = getattr(tool, "input_schema", None)

    return copy.deepcopy(schema or {
        "type": "object",
        "properties": {},
    })


def convert_tool(
    tool: Any,
    prefix: str,
) -> dict[str, Any]:
    """Convert an MCP tool into an LLM function specification.

    Prefixes such as notes__ and airbnb__ make the source server explicit
    and prevent collisions when both servers expose similarly named tools.
    """
    function_name = f"{prefix}__{tool.name}"

    return {
        "type": "function",
        "function": {
            "name": function_name,
            "description": (
                f"[{prefix} MCP server] "
                f"{tool.description or 'MCP tool'}"
            ),
            "parameters": get_input_schema(tool),
        },
    }


def available_function_names(
    functions: list[dict[str, Any]],
) -> set[str]:
    """Return the set of function names visible to the planner."""
    return {
        item["function"]["name"]
        for item in functions
    }


def extract_city(prompt: str) -> str:
    """Extract one supported demo city from natural-language text."""
    lower_prompt = prompt.lower()

    for city in ["Paris", "London", "Abidjan"]:
        if city.lower() in lower_prompt:
            return city

    # Generic fallback for phrases such as "stays in Rome".
    match = re.search(
        r"\b(?:in|near)\s+([A-Z][A-Za-zÀ-ÿ' -]{1,40})",
        prompt,
    )

    if match:
        candidate = match.group(1).strip(" .,!?")
        # Stop before common clauses.
        candidate = re.split(
            r"\s+(?:for|with|under|and|from|on)\s+",
            candidate,
            maxsplit=1,
            flags=re.IGNORECASE,
        )[0]
        return candidate.strip()

    return "Paris"


def extract_integer(
    pattern: str,
    prompt: str,
    default: int | None = None,
) -> int | None:
    """Return the first captured integer for a regex pattern."""
    match = re.search(pattern, prompt, flags=re.IGNORECASE)
    return int(match.group(1)) if match else default


def extract_note(prompt: str) -> str | None:
    """Extract a note request from the user prompt."""
    match = re.search(
        r"(?:save|add|write|remember)\s+(?:a\s+)?note"
        r"(?:\s+that|\s*:)?\s+(.+)$",
        prompt,
        flags=re.IGNORECASE,
    )

    if not match:
        return None

    return match.group(1).strip(" .")


def stub_plan(
    prompt: str,
    functions: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    """Create deterministic calls for a listing search and optional note."""
    available = available_function_names(functions)
    calls: list[dict[str, Any]] = []
    lower_prompt = prompt.lower()

    # Plan an Airbnb search for prompts related to stays or listings.
    if any(
        word in lower_prompt
        for word in [
            "airbnb",
            "stay",
            "stays",
            "listing",
            "listings",
            "apartment",
            "accommodation",
        ]
    ):
        search_name = "airbnb__airbnb_search"

        if search_name not in available:
            raise ValueError(
                "The Airbnb server did not advertise airbnb_search."
            )

        arguments: dict[str, Any] = {
            "location": extract_city(prompt),
            "adults": extract_integer(
                r"(\d+)\s+adults?",
                prompt,
                default=1,
            ),
        }

        max_price = extract_integer(
            r"(?:under|below|max(?:imum)?(?:\s+price)?(?:\s+of)?)"
            r"\s*[$€£]?\s*(\d+)",
            prompt,
        )
        limit = extract_integer(
            r"(?:find|show|give|return)\s+(\d+)\s+",
            prompt,
            default=2,
        )

        if max_price is not None:
            arguments["maxPrice"] = max_price

        # `limit` is supported by the local stub, not necessarily the real
        # OpenBnB tool. Include it only in stub mode.
        if not USE_REAL_AIRBNB:
            arguments["limit"] = limit

        calls.append({
            "name": search_name,
            "args": arguments,
        })

    note_text = extract_note(prompt)

    if note_text:
        note_name = "notes__add_note"

        if note_name not in available:
            raise ValueError(
                "The notes server did not advertise add_note."
            )

        calls.append({
            "name": note_name,
            "args": {"text": note_text},
        })

    if "list note" in lower_prompt or "show note" in lower_prompt:
        list_name = "notes__list_notes"

        if list_name in available:
            calls.append({
                "name": list_name,
                "args": {},
            })

    if not calls:
        raise ValueError(
            "The stub planner could not identify an Airbnb or notes task."
        )

    return calls


def github_models_plan(
    prompt: str,
    functions: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    """Use GitHub Models to propose prefixed function calls."""
    token = os.getenv("GITHUB_TOKEN", "").strip()

    if not token:
        raise RuntimeError(
            "Set GITHUB_TOKEN or keep USE_REAL_LLM=false."
        )

    response = requests.post(
        GITHUB_MODELS_URL,
        headers={
            "Accept": "application/vnd.github+json",
            "Authorization": f"Bearer {token}",
            "X-GitHub-Api-Version": GITHUB_API_VERSION,
            "Content-Type": "application/json",
        },
        json={
            "model": GITHUB_MODEL,
            "messages": [
                {
                    "role": "system",
                    "content": (
                        "Plan all tool calls needed for the user request. "
                        "You may use tools from both MCP servers. "
                        "Return tool calls only."
                    ),
                },
                {
                    "role": "user",
                    "content": prompt,
                },
            ],
            "tools": functions,
            "tool_choice": "auto",
            "temperature": 0,
            "max_tokens": 500,
        },
        timeout=60,
    )
    response.raise_for_status()

    message = response.json()["choices"][0]["message"]
    raw_calls = message.get("tool_calls") or []
    calls: list[dict[str, Any]] = []

    for raw_call in raw_calls:
        function = raw_call["function"]
        raw_arguments = function.get("arguments", {})
        arguments = (
            json.loads(raw_arguments)
            if isinstance(raw_arguments, str)
            else raw_arguments
        )

        calls.append({
            "name": function["name"],
            "args": arguments,
        })

    if not calls:
        raise RuntimeError("The real LLM returned no tool calls.")

    return calls


def plan_tool_calls(
    prompt: str,
    functions: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    """Select the stub or real planner."""
    if USE_REAL_LLM:
        return github_models_plan(prompt, functions)

    return stub_plan(prompt, functions)


def extract_tool_value(result: Any) -> Any:
    """Extract structured or text data from an MCP tool result."""
    structured = getattr(result, "structuredContent", None)

    if structured is None:
        structured = getattr(result, "structured_content", None)

    if isinstance(structured, dict):
        if set(structured) == {"result"}:
            return structured["result"]
        return structured

    texts: list[str] = []

    for block in getattr(result, "content", []):
        if isinstance(block, types.TextContent):
            texts.append(block.text)
            continue

        text = getattr(block, "text", None)
        if text is not None:
            texts.append(str(text))

    if len(texts) == 1:
        single = texts[0]
        try:
            return json.loads(single)
        except json.JSONDecodeError:
            return single

    return texts


def validate_call(
    call: dict[str, Any],
    prefixed_tools: dict[str, tuple[str, str]],
) -> None:
    """Reject unknown names or malformed planner arguments."""
    if not isinstance(call, dict):
        raise TypeError("Every tool call must be a dictionary.")

    name = call.get("name")
    arguments = call.get("args")

    if name not in prefixed_tools:
        raise ValueError(f"Unknown prefixed tool: {name!r}")

    if not isinstance(arguments, dict):
        raise TypeError(
            f"Arguments for {name!r} must be a dictionary."
        )


def compact_listing_summary(value: Any) -> str:
    """Create a readable deterministic summary for the stub run."""
    if not isinstance(value, dict):
        return str(value)

    listings = value.get("listings")

    if not isinstance(listings, list):
        return json.dumps(value, ensure_ascii=False)

    if not listings:
        return f"No listings found for {value.get('location', 'the location')}."

    lines = [
        f"Listings for {value.get('location', 'requested location')}:"
    ]

    for item in listings:
        lines.append(
            "- "
            f"{item.get('name')} — "
            f"{item.get('price_per_night')} per night — "
            f"rating {item.get('rating')} — "
            f"{item.get('url')}"
        )

    return "\n".join(lines)


async def orchestrate(prompt: str) -> dict[str, Any]:
    """Connect both servers, plan calls, route them, and return results."""
    notes_params = notes_server_params()
    airbnb_params = airbnb_server_params()

    async with stdio_client(notes_params) as (notes_read, notes_write):
        async with ClientSession(
            notes_read,
            notes_write,
        ) as notes_session:
            await notes_session.initialize()
            notes_tools_result = await notes_session.list_tools()

            async with stdio_client(airbnb_params) as (
                airbnb_read,
                airbnb_write,
            ):
                async with ClientSession(
                    airbnb_read,
                    airbnb_write,
                ) as airbnb_session:
                    await airbnb_session.initialize()
                    airbnb_tools_result = await airbnb_session.list_tools()

                    functions = [
                        *[
                            convert_tool(tool, "notes")
                            for tool in notes_tools_result.tools
                        ],
                        *[
                            convert_tool(tool, "airbnb")
                            for tool in airbnb_tools_result.tools
                        ],
                    ]

                    # Map each prefixed function back to its source server
                    # and original MCP tool name.
                    prefixed_tools: dict[
                        str,
                        tuple[str, str],
                    ] = {}

                    for tool in notes_tools_result.tools:
                        prefixed_tools[
                            f"notes__{tool.name}"
                        ] = ("notes", tool.name)

                    for tool in airbnb_tools_result.tools:
                        prefixed_tools[
                            f"airbnb__{tool.name}"
                        ] = ("airbnb", tool.name)

                    calls = plan_tool_calls(prompt, functions)
                    results: list[dict[str, Any]] = []

                    print("Notes tools:", [
                        tool.name
                        for tool in notes_tools_result.tools
                    ])
                    print("Airbnb tools:", [
                        tool.name
                        for tool in airbnb_tools_result.tools
                    ])
                    print("LLM/stub function names:", [
                        item["function"]["name"]
                        for item in functions
                    ])
                    print(
                        "Planner:",
                        "GitHub Models"
                        if USE_REAL_LLM
                        else "deterministic stub",
                    )
                    print("Prompt:", prompt)
                    print("tool_calls:", calls)

                    for call in calls:
                        validate_call(call, prefixed_tools)

                        prefix, original_name = prefixed_tools[
                            call["name"]
                        ]

                        session = (
                            notes_session
                            if prefix == "notes"
                            else airbnb_session
                        )

                        raw_result = await session.call_tool(
                            original_name,
                            arguments=call["args"],
                        )
                        value = extract_tool_value(raw_result)

                        record = {
                            "name": call["name"],
                            "args": call["args"],
                            "value": value,
                        }
                        results.append(record)
                        print("tool_result:", record)

                    # Produce a small deterministic final response in stub
                    # mode. The raw calls/results remain available above.
                    answer_parts: list[str] = []

                    for record in results:
                        if record["name"].startswith("airbnb__"):
                            answer_parts.append(
                                compact_listing_summary(record["value"])
                            )
                        elif record["name"] == "notes__add_note":
                            answer_parts.append(
                                f"Note: {record['value']}"
                            )
                        elif record["name"] == "notes__list_notes":
                            answer_parts.append(
                                f"Saved notes:\n{record['value']}"
                            )

                    final_answer = "\n\n".join(answer_parts)
                    print("\nFINAL ANSWER")
                    print(final_answer)

                    return {
                        "prompt": prompt,
                        "functions": functions,
                        "tool_calls": calls,
                        "tool_results": results,
                        "answer": final_answer,
                    }


def main() -> None:
    """Run the default demo from a normal terminal."""
    prompt = os.getenv(
        "DEMO_PROMPT",
        (
            "Find 2 Airbnb stays in Paris for 2 adults under $250 "
            "per night and save a note that I prefer a balcony."
        ),
    )

    asyncio.run(orchestrate(prompt))


if __name__ == "__main__":
    main()

In [ ]:
# Validate the three deliverable Python files.

for filename in [
    "local_notes_server.py",
    "airbnb_stub_server.py",
    "mcp_airbnb_agent.py",
]:
    py_compile.compile(filename, doraise=True)
    print(f"{filename} syntax: OK")

# 14. Run the standalone agent and capture output

In [ ]:
completed = subprocess.run(
    [sys.executable, "mcp_airbnb_agent.py"],
    capture_output=True,
    text=True,
    timeout=40,
    check=False,
    env={
        **os.environ,
        "USE_REAL_AIRBNB": "false",
        "USE_REAL_LLM": "false",
    },
)

print("AGENT STDOUT")
print(completed.stdout)

if completed.stderr.strip():
    print("SERVER/AGENT STDERR")
    print(completed.stderr)

print("Return code:", completed.returncode)

assert completed.returncode == 0

In [ ]:
terminal_capture = (
    "$ python mcp_airbnb_agent.py\n"
    + completed.stdout
)

Path("mcp_airbnb_terminal_capture.txt").write_text(
    terminal_capture,
    encoding="utf-8",
)

print(terminal_capture)
print("Saved: mcp_airbnb_terminal_capture.txt")

# 15. Enabling the real integrations

## Real OpenBnB Airbnb MCP server

Requirements:

- Node.js 18 or newer;
- `npx` available;
- external network access.

Set:

```python
USE_REAL_AIRBNB = True
```

The notebook starts:

```bash
npx -y @openbnb/mcp-server-airbnb
```

The real server offers `airbnb_search` and `airbnb_listing_details`. It may
contact Airbnb and third-party geocoding services. Keep robots.txt handling
enabled unless you explicitly understand and accept the test override.

## Real GitHub Models planner

Store a token with `models:read` as `GITHUB_TOKEN`, then set:

```python
USE_REAL_LLM = True
```

Stub mode remains recommended for grading because it is free and
deterministic.

# Observations

- MCP discovery allows the orchestrator to avoid hard-coding full schemas.
- Prefixes make multi-server routing explicit.
- The planner proposes actions but never receives direct Python access.
- The client validates names and arguments before execution.
- Both servers stay independent and communicate only through their own MCP
  sessions.
- Stub mode proves the architecture without depending on tokens, websites,
  or npm availability.
- In-memory notes disappear when their server process stops.

# Troubleshooting

## `mcp` command missing

Re-run the install cell and restart the notebook runtime if needed.

## `npx` command missing

Keep `USE_REAL_AIRBNB=False`, or install Node.js 18+.

## Real Airbnb connection closes

Run the server separately to inspect startup errors:

```bash
npx -y @openbnb/mcp-server-airbnb
```

## GitHub Models returns no calls

Confirm the token has `models:read`, the model supports tool calling, and
inspect the prefixed function schemas.

## Planner proposes an unknown tool

Do not execute it. The notebook's `validate_call` function intentionally
rejects names that were not discovered from either MCP server.

# Deliverables checklist

- [x] Local notes MCP server
- [x] Offline Airbnb MCP stub
- [x] Optional real OpenBnB server configuration
- [x] Two simultaneous STDIO MCP sessions
- [x] Dynamic tool discovery
- [x] Prefixed LLM function names
- [x] Stub multi-tool planner
- [x] Optional GitHub Models planner
- [x] Safe call validation
- [x] Prefix-based routing
- [x] Listing output
- [x] Saved note output
- [x] Second city test
- [x] Standalone agent script
- [x] Terminal capture
- [x] Thorough code comments and documentation

# References

- MCP Python SDK stable v1:
  https://github.com/modelcontextprotocol/python-sdk/tree/v1.x
- OpenBnB Airbnb MCP server:
  https://github.com/openbnb-org/mcp-server-airbnb
- OpenBnB npm package:
  https://www.npmjs.com/package/@openbnb/mcp-server-airbnb
- GitHub Models inference API:
  https://docs.github.com/en/rest/models/inference